<h1 style="color: #9f43c3ff; border-bottom: 3px solid #9f43c3ff; padding-bottom: 8px;">
AI & ML Course with BinX Week 8 Day 1: NLP Preprocessing & Sprint 3 Kickoff
</h1>

<blockquote style="border-left: 3px solid #2e1a9aff; padding-left: 12px; margin-left: 0;">

<b>Day 1 Learning Objectives:</b>
-  <b>Sprint 3 Planning & Backlog:</b> Establish sprint deliverables focusing on NLP and CV preprocessing, end-to-end integration, and model evaluation.
-  <b>Why Text Needs Preprocessing:</b> Understand vocabulary explosion and why raw strings must be standardized prior to vectorization.
-  <b>Tokenization Mechanics:</b> Split unstructured natural language text into discrete tokens using `nltk.word_tokenize`.
-  <b>Cleaning & Normalization Pipeline:</b> Build a modular pipeline performing lowercasing, noise/URL removal, punctuation filtering, stop-word removal, and lemmatization.
-  <b>Task-Critical Signal Preservation:</b> Audit stop-word filtering to preserve sentiment negations (`not`, `no`) and prevent polarity inversion.

</blockquote>


## <span style="color: #39b64eff">1.1 Sprint 3 Planning & Backlog Selection</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

<b>Sprint 3 Goal:</b> Transition our trained core model into an integrated, deploy-ready end-to-end pipeline (<code>predict()</code>), ensure zero training/serving skew, and execute a comprehensive evaluation combining task-appropriate metrics, error analysis, and SHAP explainability.

<b>Sprint 3 Backlog Deliverables:</b>
1. <b>Day 1 (Current):</b> NLP Preprocessing (Tokenization, Cleaning, Lemmatization vs. Stemming, Signal Preservation)
2. <b>Day 2:</b> Text Representation (TF-IDF vs. Word2Vec, GloVe, and Contextual Transformer Embeddings)
3. <b>Day 3:</b> Computer Vision Preprocessing (OpenCV, BGR vs. RGB Pitfall, Augmentation Pipeline, Mentor PR Review)
4. <b>Day 4:</b> Model Integration & Error Analysis (Unified <code>predict()</code> Pipeline, Zero Skew, Confusion Matrix Diagnostics)
5. <b>Day 5:</b> Full Evaluation & Explainability (Baseline Comparison, SMOTE / PR Tuning, SHAP Summary & Waterfall Plots)

</blockquote>


## <span style="color: #309c42ff">1.2 Environment Setup, Raw Corpus Definition & Tokenization</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

Before machine learning models can process natural language, text must be tokenized into discrete lexical units. We define our sample text corpus (including positive, technical, and negated sentences) and apply NLTK's <code>word_tokenize</code> to segment strings along whitespace and punctuation boundaries. Notice that raw tokens retain casing differences (<code>"This"</code>, <code>"The"</code>), punctuation symbols (<code>"!"</code>, <code>","</code>), and inflected verb forms that require cleaning.

</blockquote>


In [1]:
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

sample = [
    "This internship is excellent and provides structured, rigorous learning!",
    "The attention mechanism allows direct parallel access across all tokens.",
    "The baseline Logistic Regression performed adequately on tabular data.",
    "The deep model is not accurate and did not converge as expected!"
]

raw_tokens = [word_tokenize(text) for text in sample]
print("Raw Tokens (Sample 1):")
print(raw_tokens[0])
print("\nRaw Tokens (Sample 4 - Negation Sentence):")
print(raw_tokens[3])


Raw Tokens (Sample 1):
['This', 'internship', 'is', 'excellent', 'and', 'provides', 'structured', ',', 'rigorous', 'learning', '!']

Raw Tokens (Sample 4 - Negation Sentence):
['The', 'deep', 'model', 'is', 'not', 'accurate', 'and', 'did', 'not', 'converge', 'as', 'expected', '!']


## <span style="color: #309c42ff">1.3 Full Cleaning Pipeline (String & Token Operations)</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

We implement a two-phase text normalization pipeline:
1. <b>String-level cleaning:</b> Lowercases text to eliminate case redundancy, strips HTML tags, removes URLs, and standardizes whitespace.
2. <b>Token-level cleaning:</b> Tokenizes the string, strips standalone punctuation and non-alphanumeric noise, filters common English stop-words, and applies morphological lemmatization via <code>WordNetLemmatizer</code> to reduce words to their true dictionary base form.

Notice that we proactively audit the stop-word list to retain negation words (<code>"not"</code>, <code>"no"</code>) so that vital semantic signals are preserved.

</blockquote>


In [2]:
stop_words = set(stopwords.words('english'))
stop_words.discard('not')
stop_words.discard('no')
stop_words.discard('nor')

lemmatizer = WordNetLemmatizer()

def clean_and_tokenize(raw_input):
    if isinstance(raw_input, list):
        return [clean_and_tokenize(item) for item in raw_input]
    
    # 1. String-level normalization
    text = raw_input.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Tokenization
    tokens = word_tokenize(text)
    
    # 3. Token-level filtering and lemmatization
    cleaned_tokens = []
    for token in tokens:
        if token in string.punctuation or not token.isalpha():
            continue
        if token in stop_words:
            continue
        cleaned_tokens.append(lemmatizer.lemmatize(token))
        
    return cleaned_tokens

print("Single Sentence Test:")
print(clean_and_tokenize("Single sentence text here!"))

print("\nCleaned & Lemmatized Corpus:")
print(clean_and_tokenize(sample))


Single Sentence Test:
['single', 'sentence', 'text']

Cleaned & Lemmatized Corpus:
[['internship', 'excellent', 'provides', 'structured', 'rigorous', 'learning'], ['attention', 'mechanism', 'allows', 'direct', 'parallel', 'access', 'across', 'token'], ['baseline', 'logistic', 'regression', 'performed', 'adequately', 'tabular', 'data'], ['deep', 'model', 'not', 'accurate', 'not', 'converge', 'expected']]


## <span style="color: #309c42ff">1.4 Task-Critical Signal Preservation Verification</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

<b>The Preprocessing Rule:</b> Preprocessing choices are strictly task-dependent. In tasks like sentiment analysis, customer feedback classification, or medical diagnosis, removing stop words without verification can lead to catastrophic errors. Discarding <code>"not"</code> or <code>"no"</code> completely inverts the semantic meaning of a statement.

Below, we test our negated sentence against both a naive stop-word removal pipeline and our task-aware pipeline to empirically verify that crucial negation anchors survive preprocessing.

</blockquote>


In [3]:
target_sentence = sample[3]

naive_stops = set(stopwords.words('english'))
naive_tokens = [lemmatizer.lemmatize(t.lower()) for t in word_tokenize(target_sentence)
                if t.isalpha() and t.lower() not in naive_stops]

preserved_tokens = clean_and_tokenize(target_sentence)

print("Original Sentence:")
print(f'"{target_sentence}"\n')
print("Naive Cleaning Output (Lost Negation):", naive_tokens)
print("Preserved Cleaning Output (Retained) :", preserved_tokens)


Original Sentence:
"The deep model is not accurate and did not converge as expected!"

Naive Cleaning Output (Lost Negation): ['deep', 'model', 'accurate', 'converge', 'expected']
Preserved Cleaning Output (Retained) : ['deep', 'model', 'not', 'accurate', 'not', 'converge', 'expected']


## <span style="color: #9f43c3ff">1.5 Methodological Findings & Pipeline Readiness</span>

<blockquote style="border-left: 3px solid #2e1a9aff; padding-left: 12px; margin-left: 0;">

<b>Key Takeaways & Analytical Observations:</b>
1. <b>Signal Inversion Prevented:</b> In the naive pipeline, discarding <code>"not"</code> reduced <i>"The deep model is not accurate and did not converge as expected!"</i> to <code>['deep', 'model', 'accurate', 'converge', 'expected']</code>—inverting a failure report into apparent success. Our pipeline retained both negations: <code>['deep', 'model', 'not', 'accurate', 'not', 'converge', 'expected']</code>.
2. <b>Lemmatization vs. Stemming:</b> We chose <code>WordNetLemmatizer</code> because it maps inflections to valid vocabulary words (e.g., <code>"allows"</code> $\rightarrow$ <code>"allow"</code>) using morphology, avoiding the crude truncations of heuristic stemmers (e.g., Porter stemmer converting <code>"studies"</code> to <code>"studi"</code>).
3. <b>Sprint 3 Readiness:</b> The standardized, signal-preserving tokens produced today are directly formatted for feature extraction in <b>Day 2 (TF-IDF vs. Word Embeddings)</b>.

</blockquote>
